# 기본 정보 입력

In [1]:
# 패키지 불러오기
import openai
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

# 소스코드에 API Key를 직접 작성하지 않고 환경변수에서 읽습니다.
# 이렇게 하면 GitHub 등에 코드를 업로드할 때 API Key 노출 위험을 줄일 수 있습니다.
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
# API 키 지정하여 클라이언트 선언하기
client = openai.OpenAI(api_key = OPENAI_API_KEY)

In [4]:
import openai

print(openai.__version__)
print(hasattr(client.beta, "assistants"))

3.13.0
True


## 파일 업로드 및 벡터 저장소 추가하기

In [5]:
# 벡터 저장소 생성하기
vector_store = client.vector_stores.create(name="축구 규칙 파일")
# vector_store = client.beta.vector_stores.create(name="축구 규칙 파일")

In [6]:
# 파일을 벡터 저장소에 올리기
file_streams = open("축구규칙정리.pdf", "rb")

file_batch = client.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=[file_streams]
)
print(file_batch.status)
print(file_batch.file_counts)

completed
FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)


# Create Assistant
- instructions: 어시스턴트와 모델이 어떻게 행동하거나 응답해야 하는지 알려줍니다.
- model: 미세 조정된 모델을 포함하여 모든 GPT-3.5 또는 GPT-4 모델을 지정할 수 있습니다. 검색 도구에는 gpt-3.5-turbo-1106 및 gpt-4-1106-preview 모델이 필요합니다.
- tools: API는 OpenAI에서 빌드 및 호스팅하는 코드 인터프리터와 웹 검색을 지원합니다. 또한 함수 호출 기능과 유사한 동작으로 사용자 정의 함수 서명을 정의할 수 있습니다


In [7]:
instruction = '''
[목적]
이 GPT는 축구 규칙을 상세히 설명해주는 챗봇입니다.

[규칙]
1. 사용자가 축구 규칙에 대해 질문하면 업로드된 파일에서 해당 내용을 찾아 자세히 답변합니다.
2. 파일안에서 마땅한 답을 찾을 수 없거나 축구 규칙에 관한 질문이 아니면 "축구 규칙에 관한 질문만 부탁해요^^" 라고 답해주세요.
3. 답변의 형태는 아래 예시와 같이 해주세요
예시)
-  질문 : 질문 내용
-  답변 : 답변내용 
4. 모든 질문에 한국어로 답변해주세요.
'''

In [8]:
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instruction,
    input="규칙에 대해 설명해 주세요.",
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }
    ]
)

print(response.output_text)

- 질문 : 규칙에 대해 설명해 주세요.
- 답변 : 축구의 규칙은 여러 가지가 있으며, 기본적으로 경기의 진행과 페널티, 퇴장 등의 상황을 규정하고 있습니다. 예를 들어, 모든 볼은 둥글고 적절한 재질로 만들어져야 하며, 크기와 무게에도 규정이 있습니다. 또한, VAR(비디오 보조 심판) 시스템은 경기 중 명확한 실수를 보정하기 위해 사용되며, 주심의 판단을 돕는 역할을 합니다. 또 다른 규칙으로는 임시 퇴장 규정이 있으며, 선수는 경고를 받은 후 일시적으로 퇴장할 수 있습니다. 

이 외에도 다양한 규칙이 존재하므로, 특정 규칙이나 상황에 대해 더 궁금하신 점이 있으면 말씀해 주세요!


## 생성한 Assistant 업데이트 하기

In [9]:
# Run this only if you need to update the configuration of the assistant
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instruction,
    input="질문 내용",
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }
    ]
)

print(response.output_text)

축구 규칙에 관한 질문만 부탁해요^^


## 기존 Assistant 불러오기

In [10]:
# Responses API에서 사용할 Vector Store 목록 확인
vector_stores = client.vector_stores.list(limit=20)
for store in vector_stores.data:
    print(f"{store.id}: {store.name} ({store.status})")

vs_6aa39ede7b548191b363941313b72fb1: 축구 규칙 파일 (completed)
vs_6aa397e39d1c819188c0af2cd9a5e70c: 축구 규칙 파일 (completed)
vs_6aa394148194819197a27b81145b358e: 축구 규칙 파일 (completed)


In [11]:
store = client.vector_stores.retrieve(vector_store.id)
print(store)

VectorStore(id='vs_6aa39ede7b548191b363941313b72fb1', created_at=1789107934, file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1), last_active_at=1789107946, metadata={}, name='축구 규칙 파일', object='vector_store', status='completed', usage_bytes=370870, expires_after=None, expires_at=None, description=None)


# Thread

## Thread 생성하기

In [ ]:
# Responses API는 별도의 Thread 객체가 필요하지 않습니다.

# 결과확인(Run complete 후)

In [12]:
question="축구장의 크기는?"
response=client.responses.create(model="gpt-4o-mini",instructions=instruction,input=question,tools=[{"type":"file_search","vector_store_ids":[vector_store.id]}])
print(response.output_text)


- 질문 : 축구장의 크기는?
- 답변 : 축구장의 크기는 다음과 같습니다:
  - 터치라인의 길이는 최소 90m(100 야드)에서 최대 120m(130 야드)입니다.
  - 골라인의 길이는 최소 45m(50 야드)에서 최대 90m(100 야드)입니다.
  - 국제 경기를 위한 크기는 터치라인 최소 100m(110 야드)에서 최대 110m(120 야드)이며, 골라인은 최소 64m(70 야드)에서 최대 75m(80 야드)입니다.


In [13]:
# Responses API에서는 client.responses.create()가 요청과 실행을 함께 처리합니다.
print(response.status)

completed


# File 삭제

In [17]:
files=client.vector_stores.files.list(vector_store_id=vector_store.id)
for item in files.data: print(item.id,item.status)


file-AjNDxL5rMuutgUHwUBiFJ4 completed


In [18]:
# 특정 파일 ID 가져오기
file_id = files.data[0].id
print(file_id)

file-AjNDxL5rMuutgUHwUBiFJ4


In [ ]:
# Vector Store 파일 삭제는 별도 관리 코드에서 수행합니다.
